# Java Repository Analysis

This notebook analyzes the LOC output produced by `scripts/github_repo_java_loc_analysis.py`.

Recommended launch command from the repository root:

```bash
source .venv/bin/activate
jupyter lab
```

The thresholds are configurable in the next cell.

In [36]:
from pathlib import Path
import json
from html import escape
from IPython.display import HTML, display

DATA_PATH = Path("../java-repos-loc.json")
MIN_TOTAL_LOC = 1500
MAX_TOTAL_LOC = 7000
ONLY_SUCCESSFUL_ROWS = True

print(f"Using analysis file: {DATA_PATH}")
print(f"LOC range: {MIN_TOTAL_LOC} to {MAX_TOTAL_LOC}")

Using analysis file: ../java-repos-loc.json
LOC range: 1500 to 7000


In [37]:
payload = json.loads(DATA_PATH.read_text(encoding="utf-8"))
repositories = payload["repositories"] if isinstance(payload, dict) else payload

if ONLY_SUCCESSFUL_ROWS:
    repositories = [repo for repo in repositories if repo.get("status") == "ok"]

print(f"Loaded {len(repositories)} repositories")
if isinstance(payload, dict) and "summary" in payload:
    print("Existing script summary:")
    print(json.dumps(payload["summary"], indent=2))

Loaded 110 repositories
Existing script summary:
{
  "repositories_analyzed": 110,
  "successful": 110,
  "errors": 0,
  "java_main_language_count": 92,
  "java_main_language_share": 0.836364,
  "aggregate_java_loc": 603099,
  "aggregate_total_loc": 1029851,
  "aggregate_java_share": 0.585618
}


In [38]:
def as_int(value):
    return int(value or 0)


def in_loc_range(repo, minimum, maximum):
    total_loc = as_int(repo.get("java_loc"))
    return minimum <= total_loc <= maximum


repos_in_range = [repo for repo in repositories if in_loc_range(repo, MIN_TOTAL_LOC, MAX_TOTAL_LOC)]
repos_java_main = [repo for repo in repositories if repo.get("java_is_main_language")]
repos_in_range_java_main = [repo for repo in repos_in_range if repo.get("java_is_main_language")]

summary = {
    "total_repositories": len(repositories),
    "repos_with_total_loc_in_range": len(repos_in_range),
    "repos_with_java_as_main_language": len(repos_java_main),
    "repos_in_range_with_java_as_main_language": len(repos_in_range_java_main),
}

summary

{'total_repositories': 110,
 'repos_with_total_loc_in_range': 53,
 'repos_with_java_as_main_language': 92,
 'repos_in_range_with_java_as_main_language': 48}

In [39]:
def display_rows(rows, columns, limit=None):
    subset = rows if limit is None else rows[:limit]
    header_html = "".join(f"<th>{escape(str(column))}</th>" for column in columns)
    body_html = []
    for row in subset:
        cells = "".join(f"<td>{escape(str(row.get(column, '')))}</td>" for column in columns)
        body_html.append(f"<tr>{cells}</tr>")
    table_html = (
        "<table>"
        f"<thead><tr>{header_html}</tr></thead>"
        f"<tbody>{''.join(body_html)}</tbody>"
        "</table>"
    )
    display(HTML(table_html))


columns = ["name", "total_loc", "java_loc", "java_percent", "java_is_main_language", "top_language"]

print("Repositories with total LOC inside the selected range:")
display_rows(sorted(repos_in_range, key=lambda repo: as_int(repo.get("total_loc"))), columns)


Repositories with total LOC inside the selected range:


name,total_loc,java_loc,java_percent,java_is_main_language,top_language
EsotericSoftware/reflectasm,1920,1692,88.12,True,Java
LeonardoZ/java-concurrency-patterns,1977,1884,95.3,True,Java
whwlsfb/JDumpSpider,2663,2341,87.91,True,Java
alamkanak/Android-Week-View,2808,1963,69.91,True,Java
YeautyYE/netty-websocket-spring-boot-starter,2859,2102,73.52,True,Java
JakeWharton/RxRelay,2972,2405,80.92,True,Java
esoxjem/MovieGuide,3326,2342,70.41,True,Java
monkeyWie/proxyee,3440,2755,80.09,True,Java
mbechler/marshalsec,3526,3185,90.33,True,Java
pedrovgs/EffectiveAndroidUI,3541,2225,62.84,True,Java


In [40]:
print("Repositories in range where Java is the main language:")
display_rows(
    sorted(repos_in_range_java_main, key=lambda repo: as_int(repo.get("total_loc"))),
    ["name", "total_loc", "java_loc", "java_percent", "top_language"],
)


Repositories in range where Java is the main language:


name,total_loc,java_loc,java_percent,top_language
EsotericSoftware/reflectasm,1920,1692,88.12,Java
LeonardoZ/java-concurrency-patterns,1977,1884,95.3,Java
whwlsfb/JDumpSpider,2663,2341,87.91,Java
alamkanak/Android-Week-View,2808,1963,69.91,Java
YeautyYE/netty-websocket-spring-boot-starter,2859,2102,73.52,Java
JakeWharton/RxRelay,2972,2405,80.92,Java
esoxjem/MovieGuide,3326,2342,70.41,Java
monkeyWie/proxyee,3440,2755,80.09,Java
mbechler/marshalsec,3526,3185,90.33,Java
pedrovgs/EffectiveAndroidUI,3541,2225,62.84,Java


In [41]:
import csv

export_path = DATA_PATH.with_name(
    f"{DATA_PATH.stem.replace('-loc', '')}-java-main-{MIN_TOTAL_LOC}-{MAX_TOTAL_LOC}.csv"
)

with export_path.open("w", newline="", encoding="utf-8") as handle:
    writer = csv.writer(handle)
    for repo in sorted(repos_in_range_java_main, key=lambda repo: repo["name"].lower()):
        writer.writerow([repo["name"]])

print(f"Wrote {len(repos_in_range_java_main)} repository names to {export_path}")


Wrote 48 repository names to ../java-repos-java-main-1500-7000.csv


In [50]:
BASE_COMPARE_PATH = Path("../java-repos-loc.json")
ADD_COMPARE_PATH = Path("../java-repos-sup2-loc.json")

def load_repo_rows(path):
    payload = json.loads(path.read_text(encoding="utf-8"))
    repos = payload["repositories"] if isinstance(payload, dict) else payload
    return [repo for repo in repos if repo.get("status") == "ok" and "name" in repo]

base_compare_rows = load_repo_rows(BASE_COMPARE_PATH)
add_compare_rows = load_repo_rows(ADD_COMPARE_PATH)
base_repo_names = {repo["name"] for repo in base_compare_rows}
new_compare_rows = [repo for repo in add_compare_rows if repo["name"] not in base_repo_names]
new_repo_rows_in_loc_range = [
    repo for repo in new_compare_rows
    if in_loc_range(repo, MIN_TOTAL_LOC, MAX_TOTAL_LOC)
]
new_repo_names_in_loc_range = sorted(repo["name"] for repo in new_repo_rows_in_loc_range)

print(f"Base comparison file: {BASE_COMPARE_PATH}")
print(f"Add-on comparison file: {ADD_COMPARE_PATH}")
print(f"Raw new repositories present in add-on but not in base: {len(new_compare_rows)}")
print(f"New repositories inside the selected LOC range: {len(new_repo_names_in_loc_range)}")
new_repo_names_in_loc_range


Base comparison file: ../java-repos-loc.json
Add-on comparison file: ../java-repos-sup2-loc.json
Raw new repositories present in add-on but not in base: 21
New repositories inside the selected LOC range: 7


['JMCuixy/swagger2word',
 'eugene-khyst/postgresql-event-sourcing',
 'facebook/SoLoader',
 'feiniaojin/graceful-response',
 'jenkinsci/jenkinsfile-runner',
 'mitre/HTTP-Proxy-Servlet',
 'stealthcopter/AndroidNetworkTools']

In [51]:
new_repos_export_path = ADD_COMPARE_PATH.with_name(
    f"{ADD_COMPARE_PATH.stem.replace('-loc', '')}-only-new-in-loc-range-vs-{BASE_COMPARE_PATH.stem.replace('-loc', '')}.csv"
)

with new_repos_export_path.open("w", newline="", encoding="utf-8") as handle:
    writer = csv.writer(handle)
    for repo_name in new_repo_names_in_loc_range:
        writer.writerow([repo_name])

print(f"Wrote {len(new_repo_names_in_loc_range)} new repository names to {new_repos_export_path}")


Wrote 7 new repository names to ../java-repos-sup2-only-new-in-loc-range-vs-java-repos.csv


In [ ]:
import os
import urllib.error
import urllib.parse
import urllib.request

LATEX_TABLE_ROWS = repos_in_range_java_main
QUERY_SOURCE_LABEL = "Secondary"
METADATA_EXPORT_PATH = DATA_PATH.with_name(f"{DATA_PATH.stem.replace('-loc', '')}.json")
LATEX_OUTPUT_PATH = DATA_PATH.with_name(
    f"{DATA_PATH.stem.replace('-loc', '')}-java-main-{MIN_TOTAL_LOC}-{MAX_TOTAL_LOC}.tex"
)
LATEX_CAPTION = "Final benchmark repositories."
LATEX_LABEL = "tab:final_benchmark_repositories"
GITHUB_TOKEN = ""

def load_export_metadata(path):
    if not path.exists():
        return {}
    payload = json.loads(path.read_text(encoding="utf-8"))
    repos = payload["repositories"] if isinstance(payload, dict) else payload
    return {repo["name"]: repo for repo in repos if "name" in repo}

def fetch_github_repo_metadata(repo_name):
    url = f"https://api.github.com/repos/{urllib.parse.quote(repo_name, safe='/')}"
    headers = {
        "Accept": "application/vnd.github+json",
        "X-GitHub-Api-Version": "2026-03-10",
        "User-Agent": "java-repo-analysis-notebook",
    }
    if GITHUB_TOKEN:
        headers["Authorization"] = f"Bearer {GITHUB_TOKEN}"
    request = urllib.request.Request(url, headers=headers)
    with urllib.request.urlopen(request) as response:
        return json.load(response)

metadata_by_repo = load_export_metadata(METADATA_EXPORT_PATH)
table_rows = []

for repo in LATEX_TABLE_ROWS:
    repo_name = repo["name"]
    repo_meta = metadata_by_repo.get(repo_name, {})
    stars = repo_meta.get("stars")
    forks = repo_meta.get("forks")
    description = repo_meta.get("description")
    if stars is None or forks is None or description is None:
        github_meta = fetch_github_repo_metadata(repo_name)
        stars = github_meta.get("stargazers_count", stars)
        forks = github_meta.get("forks_count", forks)
        description = github_meta.get("description") or description or ""
    table_rows.append(
        {
            "name": repo_name,
            "stars": int(stars or 0),
            "forks": int(forks or 0),
            "java_loc": int(repo.get("java_loc", 0)),
            "query_source": QUERY_SOURCE_LABEL,
            "description": description or "",
        }
    )

print(f"Prepared {len(table_rows)} rows for LaTeX export")
print(f"Metadata export source: {METADATA_EXPORT_PATH}")
print(f"GitHub token present: {bool(GITHUB_TOKEN)}")


Prepared 48 rows for LaTeX export
Metadata export source: ../java-repos.json
GitHub token present: True


In [ ]:
from pathlib import Path
import csv
import json
import os
import urllib.parse
import urllib.request

REPO_LIST_PATH = Path("../java-repos-sup2-only-new-in-loc-range-vs-java-repos.csv")
LOC_SOURCE_PATH = Path("../java-repos-sup2-loc.json")
METADATA_EXPORT_PATH = Path("../java-repos-sup2.json")
LATEX_OUTPUT_PATH = Path("../java-repos-sup2-only-new.tex")

QUERY_SOURCE_LABEL = "Primary"
LATEX_CAPTION = "Final benchmark repositories."
LATEX_LABEL = "tab:final_benchmark_repositories"

GITHUB_TOKEN = ""

def load_csv_repo_names(path):
    with path.open("r", encoding="utf-8", newline="") as handle:
        return [row[0].strip() for row in csv.reader(handle) if row and row[0].strip()]

def load_loc_rows(path):
    payload = json.loads(path.read_text(encoding="utf-8"))
    repos = payload["repositories"] if isinstance(payload, dict) else payload
    return [repo for repo in repos if repo.get("status") == "ok" and "name" in repo]

def load_export_metadata(path):
    if not path.exists():
        return {}
    payload = json.loads(path.read_text(encoding="utf-8"))
    repos = payload["repositories"] if isinstance(payload, dict) else payload
    return {repo["name"]: repo for repo in repos if "name" in repo}

def fetch_github_repo_metadata(repo_name):
    url = f"https://api.github.com/repos/{urllib.parse.quote(repo_name, safe='/')}"
    headers = {
        "Accept": "application/vnd.github+json",
        "X-GitHub-Api-Version": "2026-03-10",
        "User-Agent": "java-repo-analysis-notebook",
    }
    if GITHUB_TOKEN:
        headers["Authorization"] = f"Bearer {GITHUB_TOKEN}"
    request = urllib.request.Request(url, headers=headers)
    with urllib.request.urlopen(request) as response:
        return json.load(response)

target_repo_names = set(load_csv_repo_names(REPO_LIST_PATH))
loc_rows = load_loc_rows(LOC_SOURCE_PATH)
LATEX_TABLE_ROWS = [repo for repo in loc_rows if repo["name"] in target_repo_names]

metadata_by_repo = load_export_metadata(METADATA_EXPORT_PATH)

table_rows = []
for repo in LATEX_TABLE_ROWS:
    repo_name = repo["name"]
    repo_meta = metadata_by_repo.get(repo_name, {})
    stars = repo_meta.get("stars")
    forks = repo_meta.get("forks")
    description = repo_meta.get("description")

    if stars is None or forks is None or description is None:
        github_meta = fetch_github_repo_metadata(repo_name)
        stars = github_meta.get("stargazers_count", stars)
        forks = github_meta.get("forks_count", forks)
        description = github_meta.get("description") or description or ""

    table_rows.append({
        "name": repo_name,
        "stars": int(stars or 0),
        "forks": int(forks or 0),
        "java_loc": int(repo.get("java_loc", 0)),
        "query_source": QUERY_SOURCE_LABEL,
        "description": description or "",
    })

print(f"Prepared {len(table_rows)} rows")


Prepared 7 rows


In [53]:
def latex_escape(value):
    text = str(value)
    replacements = {
        "\\": r"\\textbackslash{}",
        "&": r"\\&",
        "%": r"\\%",
        "$": r"\\$",
        "#": r"\\#",
        "_": r"\\_",
        "{": r"\\{",
        "}": r"\\}",
        "~": r"\\textasciitilde{}",
        "^": r"\\textasciicircum{}",
    }
    for old, new in replacements.items():
        text = text.replace(old, new)
    return text

def latex_repo_cell(name, description):
    escaped_name = latex_escape(name)
    escaped_description = latex_escape(description).strip()
    if escaped_description:
        return rf"\textbf{{{escaped_name}}}\newline \textit{{{escaped_description}}}"
    return rf"\textbf{{{escaped_name}}}"

latex_lines = [
    r"\begin{longtable}{p{0.34\textwidth} p{0.13\textwidth} p{0.13\textwidth} p{0.16\textwidth} p{0.14\textwidth}}",
    f"\caption{{{latex_escape(LATEX_CAPTION)}}}",
    f"\label{{{latex_escape(LATEX_LABEL)}}}\\\\",
    r"\hline",
    "\textbf{Repository} & \textbf{Stars} & \textbf{Forks} & \textbf{Java LOC} & \textbf{Query source} \\\\",
    r"\hline",
    r"\endfirsthead",
    "",
    r"\hline",
    "\textbf{Repository} & \textbf{Stars} & \textbf{Forks} & \textbf{Java LOC} & \textbf{Query source} \\\\",
    r"\hline",
    r"\endhead",
    "",
]

for row in table_rows:
    latex_lines.append(
        f"{latex_repo_cell(row['name'], row['description'])} & {row['stars']} & {row['forks']} & {row['java_loc']} & {latex_escape(row['query_source'])} \\\\"
    )

latex_lines.extend([r"\hline", r"\end{longtable}"])
latex_table = "\n".join(latex_lines)
LATEX_OUTPUT_PATH.write_text(latex_table + "\n", encoding="utf-8")
print(latex_table)
print(f"Wrote LaTeX table to {LATEX_OUTPUT_PATH}")


\begin{longtable}{p{0.34\textwidth} p{0.13\textwidth} p{0.13\textwidth} p{0.16\textwidth} p{0.14\textwidth}}
\caption{Final benchmark repositories.}
\label{tab:final\\_benchmark\\_repositories}\\
\hline
	extbf{Repository} & 	extbf{Stars} & 	extbf{Forks} & 	extbf{Java LOC} & 	extbf{Query source} \\
\hline
\endfirsthead

\hline
	extbf{Repository} & 	extbf{Stars} & 	extbf{Forks} & 	extbf{Java LOC} & 	extbf{Query source} \\
\hline
\endhead

\textbf{mitre/HTTP-Proxy-Servlet}\newline \textit{Smiley's HTTP Proxy implemented as a Java servlet} & 1486 & 562 & 1788 & Primary \\
\textbf{stealthcopter/AndroidNetworkTools}\newline \textit{Set of useful android network tools} & 1483 & 297 & 1673 & Primary \\
\textbf{facebook/SoLoader}\newline \textit{Native code loader for Android} & 1432 & 190 & 5693 & Primary \\
\textbf{eugene-khyst/postgresql-event-sourcing}\newline \textit{A reference implementation of an event-sourced system that uses PostgreSQL as an event store built with Spring Boot. Fork th

<>:28: SyntaxWarning: "\c" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\c"? A raw string is also an option.
<>:29: SyntaxWarning: "\l" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\l"? A raw string is also an option.
<>:28: SyntaxWarning: "\c" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\c"? A raw string is also an option.
<>:29: SyntaxWarning: "\l" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\l"? A raw string is also an option.
/var/folders/qm/cz_dk_r10sb988kffn_x061h0000gn/T/ipykernel_1624/2992192085.py:28: SyntaxWarning: "\c" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\c"? A raw string is also an option.
  f"\caption{{{latex_escape(LATEX_CAPTION)}}}",
/var/folders/qm/cz_dk_r10sb988kffn_x061h0000gn/T/ipykernel_1624/2992192085.py:29: SyntaxWarning: "\l" is an